# RSO-901 Glycol Set Points — Monthly

Chiller glycol **set point** changes over one `month_obs`, from the
`lsst.sal.HVAC.logevent_chillerConfiguration` event. Each of the four HVAC chillers gets its own
panel (single column, **linked x-axis** so zooming/panning one zooms all — handy when a chiller has
many changes). A set point is piecewise-constant, so it is drawn as a **step** function with the area
beneath it filled; genuine changes are also listed in a change-log table.

See `ai_notes.md` in this folder for the topic survey and field reference.

**`month_obs`** is an integer `YYYYMM` (e.g. `202601`). Following the Rubin day-rollover convention
(the observatory day rolls at **UTC−12**, i.e. 12:00 UTC — close to, but not exactly, Chilean local
noon), the window **starts at 12:00 on the first day of the month** and **ends at 11:59 on the last
day of the month**. That excludes the final night of the month; flip `INCLUDE_LAST_NIGHT` below to
extend to the following noon.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import calendar
import colorsys

import astropy.units as u
import pandas as pd

from bokeh.io import output_notebook, output_file, save, show
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, Range1d
from bokeh.plotting import figure

from lsst.summit.utils.efdUtils import (
    getEfdData,
    makeEfdClient,
    getDayObsStartTime,
)

output_notebook()

## Parameters

In [ ]:
# Month to analyse, as YYYYMM.
month_obs = 202601

# Include the last night of the month (extend end to the following noon)?
INCLUDE_LAST_NIGHT = False

TOPIC = "lsst.sal.HVAC.logevent_chillerConfiguration"

# device_id (DeviceId_chiller enum) -> (name, colour).
# Bright, saturated palette so the four step lines stay distinct on a shared panel.
CHILLERS = {
    101: ("chiller01", "#00C2C2"),  # teal
    102: ("chiller02", "#FF7A00"),  # orange
    103: ("chiller03", "#B026D9"),  # purple
    104: ("chiller04", "#1E90FF"),  # blue
}

In [ ]:
year, month = divmod(month_obs, 100)
n_days = calendar.monthrange(year, month)[1]

first_day_obs = month_obs * 100 + 1
last_day_obs = month_obs * 100 + n_days

begin = getDayObsStartTime(first_day_obs)  # first day 12:00 UTC
if INCLUDE_LAST_NIGHT:
    end = getDayObsStartTime(last_day_obs) + 24 * u.hour  # following noon
else:
    end = getDayObsStartTime(last_day_obs) - 1 * u.s  # last day 11:59:59 UTC

print(f"month_obs {month_obs}: {begin.isot} -> {end.isot} (UTC)")

## Query the set-point (re)configuration events

This is a sparse event — a row is written only when a chiller's set point is (re)configured.

In [ ]:
df = getEfdData(efd_client := makeEfdClient(), TOPIC, begin=begin, end=end)
print(f"{len(df)} configuration event(s) for month_obs {month_obs}")
df[["device_id", "activeSetpoint"]].head() if len(df) else df

## Figure — all chillers on a single panel

All four chillers are drawn as **step** lines on one shared panel, one colour each. A set point is
piecewise-constant, so the last value is held to the end of the window. Use the box-zoom /
wheel-zoom / pan tools to inspect clusters of changes.

In [ ]:
# Naive UTC datetimes for Bokeh (its datetime axis assumes UTC).
x_lo = begin.utc.datetime
x_hi = end.utc.datetime


def build_figure():
    """Build the single-panel step plot from fresh Bokeh models.

    Bokeh models can belong to only one document, so `save(...)` (file doc) and
    `show(...)` (notebook doc) each need their own set of objects — hence a builder.
    """
    p = figure(
        height=420,
        sizing_mode="stretch_width",
        x_axis_type="datetime",
        x_range=Range1d(start=x_lo, end=x_hi),
        title=f"HVAC chiller active set points — month_obs {month_obs}",
        tools="xpan,xwheel_zoom,box_zoom,reset,save",
        active_scroll="xwheel_zoom",
    )

    for dev_id, (name, color) in CHILLERS.items():
        g = df[df["device_id"] == dev_id].sort_index() if len(df) else df
        if not len(g):
            continue
        # Drop tz (values are UTC) and hold the last value to the end of the window.
        gi = g.index
        gi = gi.tz_localize(None) if gi.tz is not None else gi
        xs = list(gi) + [x_hi]
        ys = list(g["activeSetpoint"]) + [g["activeSetpoint"].iloc[-1]]
        src = ColumnDataSource(dict(x=xs, y=ys))
        step = p.step(
            x="x", y="y", source=src, mode="after",
            line_color=color, line_width=2, legend_label=name,
        )
        p.add_tools(
            HoverTool(
                renderers=[step],
                tooltips=[
                    ("chiller", name),
                    ("time", "@x{%F %T}"),
                    ("set point", "@y{0.00} °C"),
                ],
                formatters={"@x": "datetime"},
                mode="mouse",
            )
        )

    p.xaxis.axis_label = "Time [UTC]"
    p.yaxis.axis_label = "Set point [°C]"
    p.legend.location = "top_left"
    p.legend.click_policy = "hide"
    return p


os.makedirs("plots", exist_ok=True)
output_file(f"plots/chiller_setpoints_{month_obs}.html", title=f"Chiller set points {month_obs}")
save(build_figure())  # fresh models -> file document

output_notebook()  # switch rendering back to inline
show(build_figure())  # fresh models -> notebook document

## Change-log table

Keep only rows where the set point actually moved (per chiller). This also drops redundant
re-publications (e.g. a CSC restart re-emits the current config unchanged).

In [ ]:
if len(df):
    changes = (
        df.sort_index()
        .groupby("device_id")["activeSetpoint"]
        .apply(lambda s: s[s.diff().ne(0)])  # first row + genuine changes only
        .reset_index()
        .rename(columns={"level_1": "time"})
    )
    changes["chiller"] = changes["device_id"].map(lambda d: CHILLERS[d][0])
    changes["delta_C"] = changes.groupby("device_id")["activeSetpoint"].diff()
    display(changes[["time", "chiller", "activeSetpoint", "delta_C"]])
else:
    print("No configuration events for this month_obs.")